# TrendCast — Starter Notebook

Loads the synthetic engagement-trends dataset, explores a few series, and reproduces the **naive / drift / TrendCast** benchmark table (40 series, horizon = 14).

Dataset: `trends_dataset.csv` — 40 series x 120 days, archetypes `viral_spike`, `slow_burn`, `seasonal`, `fad_fade`.

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

# On Kaggle the file lands under /kaggle/input/<dataset-slug>/
import glob, os
path = next(iter(glob.glob('/kaggle/input/**/trends_dataset.csv', recursive=True)
                 + glob.glob('trends_dataset.csv') + glob.glob('data/trends_dataset.csv')), 'trends_dataset.csv')
df = pd.read_csv(path)
print(path)
df.head()

In [ ]:
print('rows:', len(df), '| series:', df.series_id.nunique(), '| days/series:', df.day.max()+1)
df.archetype.value_counts()

## One series per archetype

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 6), sharex=True)
for ax, arch in zip(axes.ravel(), ['viral_spike','slow_burn','seasonal','fad_fade']):
    sid = df[df.archetype==arch].series_id.iloc[0]
    s = df[df.series_id==sid]
    ax.plot(s.day, s.value); ax.set_title(f'{arch}  ({sid})')
plt.tight_layout(); plt.show()

## Reproduce the benchmark
Baselines are self-contained; TrendCast uses the bundled package if present.

In [ ]:
def mae(a,b): return float(np.mean(np.abs(np.asarray(a)-np.asarray(b))))
def smape(a,b):
    a,b=np.asarray(a,float),np.asarray(b,float)
    return float(np.mean(200*np.abs(a-b)/(np.abs(a)+np.abs(b)+1e-9)))
def naive(train,h): return np.repeat(train[-1],h)
def drift(train,h,roll=7):
    w=train[-roll:]; slope=np.polyfit(np.arange(len(w)),w,1)[0]
    return train[-1]+slope*np.arange(1,h+1)

try:
    from trendcast.model import TrendCast
    have_tc=True
except Exception as e:
    have_tc=False; print('TrendCast package not importable here:', e)

H=14; models=['naive','drift']+(['trendcast'] if have_tc else [])
sm={m:[] for m in models}; ma={m:[] for m in models}
for sid,g in df.sort_values('day').groupby('series_id'):
    s=g.value.to_numpy()
    if len(s)<=H+10: continue
    tr,ac=s[:-H],s[-H:]
    preds={'naive':naive(tr,H),'drift':drift(tr,H)}
    if have_tc: preds['trendcast']=TrendCast().fit(tr).forecast(H).point
    for m in models: sm[m].append(smape(ac,preds[m])); ma[m].append(mae(ac,preds[m]))

print(f"{'model':<12}{'sMAPE %':>10}{'MAE':>10}")
for m in models: print(f'{m:<12}{np.mean(sm[m]):>10.2f}{np.mean(ma[m]):>10.2f}')